In [1]:
import pandas as pd
import glob

In [2]:
import numpy as np
import json
import os

STATS_PATH = "C:/Users/Gabriel/Documents/Projects/Global-Tension-Monitor/data/features/stats/stats_referencia.json"
MIN_DIAS_PARA_CONGELAR = 30

def calcular_ou_carregar_stats(df):
    '''Calcula ou carrega os stats de referência para o WTI'''
    dias_disponiveis = df['date'].nunique()

    if os.path.exists(STATS_PATH):
        with open(STATS_PATH) as f:
            return json.load(f)

    if dias_disponiveis < MIN_DIAS_PARA_CONGELAR:
        print(f"[AVISO] Apenas {dias_disponiveis} dias — stats provisórios - régua ainda instável")
        congelar = False
    else:
        congelar = True

    media_log = np.log1p(
        df['mentions'] * 0.5 +
        df['articles'] * 0.3 +
        df['sources']  * 0.2
    )
    stats = {
        "p5_media":  media_log.quantile(0.05),
        "p95_media": media_log.quantile(0.95),
        "p5_wti":    None,  # preenchido depois do primeiro calcular_wti completo
        "p95_wti":   None
    }

    if congelar:
        with open(STATS_PATH, "w") as f:
            json.dump(stats, f)
        print(f"[INFO] Stats congelados com {dias_disponiveis} dias de histórico")

    return stats


In [ ]:
def eng_feat(df):
    '''Engenharia de Features para o WTI e variáveis temporais'''
    df = df.copy()

    stats = calcular_ou_carregar_stats(df)

    # Goldstein
    df['goldstein_norm'] = (10 - df['goldstein']) / 20

    # Tone 
    df['tone_norm'] = df['tone'].clip(upper=0).abs() / 100

    # Media weight 
    media_log = np.log1p(
        df['mentions'] * 0.5 +
        df['articles'] * 0.3 +
        df['sources']  * 0.2
    )
    p5_m  = stats["p5_media"]
    p95_m = stats["p95_media"]

    df['media_weight'] = (
        (media_log - p5_m) / (p95_m - p5_m)
    ).clip(0, 1)

    # WTI raw 
    df['wti_raw'] = (
        df['goldstein_norm'] * 0.6 +
        df['tone_norm']      * 0.2 +
        df['media_weight']   * 0.2
    )

    # WTI final e princpal feature
    if stats["p5_wti"] is None:
        w5  = df['wti_raw'].quantile(0.05)
        w95 = df['wti_raw'].quantile(0.95)
    else:
        w5  = stats["p5_wti"]
        w95 = stats["p95_wti"]

    df['wti'] = (
        (df['wti_raw'] - w5) / (w95 - w5)
    ).clip(0, 1)

    # Congela stats
    if stats["p5_wti"] is None and os.path.exists(STATS_PATH):
        stats["p5_wti"]  = df['wti_raw'].quantile(0.05)
        stats["p95_wti"] = df['wti_raw'].quantile(0.95)
        with open(STATS_PATH, "w") as f:
            json.dump(stats, f)

    # 2. ADIÇÃO DAS VARIÁVEIS TEMPORAIS  

    # Deltas
    df['wti_delta_1d'] = df.groupby('actor1_country')['wti'].diff()
    df['tone_delta_1d'] = df.groupby('actor1_country')['tone_norm'].diff()

    # Janelas Móveis (3 dias)
    df['wti_mean_3d'] = df.groupby('actor1_country')['wti'].transform(
        lambda x: x.rolling(window=3, min_periods=1).mean()
    )
    
    df['tone_std_3d'] = df.groupby('actor1_country')['tone_norm'].transform(
        lambda x: x.rolling(window=3, min_periods=2).std()
    )

    # Proteção contra Gaps de tempo
    df['gap'] = df.groupby('actor1_country')['date'].diff().dt.days
    df.loc[df['gap'] > 1, ['wti_delta_1d', 'tone_delta_1d']] = np.nan

    #3. CRIAÇÃO DO TARGET
    
    # Criação dos níveis de hoje baseados no WTI
    # 0: Baixa (WTI < 0.4)  1: Média (0.4 <= WTI < 0.7)  2: Alta (WTI >= 0.7)

    df['nivel_crise_hoje'] = pd.cut(
        df['wti'], 
        bins=[-np.inf, 0.4, 0.7, np.inf], 
        labels=[0, 1, 2]
    )


    df['nivel_crise_hoje'] = df['nivel_crise_hoje'].astype(float).fillna(-1).astype(int)
    
    # Deslocamos o nível de amanhã para a linha de hoje
    df['target_crise'] = df.groupby('actor1_country')['nivel_crise_hoje'].shift(-1)

    # Validação de data: só mantemos o gabarito se o registro de amanhã for realmente o dia seguinte
    df['future_date_check'] = df.groupby('actor1_country')['date'].shift(-1)
    mask_invalid = (df['future_date_check'] - df['date']).dt.days != 1
    df.loc[mask_invalid, 'target_crise'] = np.nan

    #adionando contexto temporal
    df['dia_da_semana'] = df['date'].dt.dayofweek
    df['mes'] = df['date'].dt.month
    # Limpeza final das colunas auxiliares
    df = df.drop(columns=['gap', 'future_date_check', 'nivel_crise_hoje'])
    
    #deixando mais leve
    colunas_float = df.select_dtypes(include=['float64']).columns
    df[colunas_float] = df[colunas_float].astype('float32')

    return df

In [12]:
import pandas as pd

# Carrega o arquivo consolidado vindo da coleta nova
df_treino = pd.read_parquet("C:/Users/Gabriel/Documents/Projects/Global-Tension-Monitor/data/processed/historico_agrupado_10anos.parquet")

# Ordena para garantir o funcionamento correto das janelas móveis (rolling)
df_treino = df_treino.sort_values(["actor1_country", "date"])

# Executa toda a sua lógica de pesos, normalização e WTI localmente
df_treino = eng_feat(df_treino)

# Dropa os nulos do target para o treino e salva a base final do modelo
df_treino = df_treino.dropna(subset=['target_crise'])
df_treino.to_parquet("C:/Users/Gabriel/Documents/Projects/Global-Tension-Monitor/data/features/gold.parquet", index=False)

Timestamp('2016-01-13 00:00:00')